# P17 store written without a checker
Silent Failures lessons rewritten from the same 40 runs without the evaluator's outcome ('nf'), labelled before any guide or agent run. Arms: C1-nf (all own nf lessons), E-nf (embedding top-3), G3-nf (P13 guide), G3d-nf (P16 data-profile guide). Predictions: `quick_runs/predictions/P17.md`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
import re
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
SF6 = TASKS4 + ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
FIX = {"sf_tf05_credit_card_leakage": {164, 166, 167, 168, 169}, "sf_tf03_correlation_causation": {127, 129, 131, 132, 134, 135, 137}}
CONTRA3 = {130, 136}; WRONG_IDS = set(range(708, 720))
INSTR = r"task (says|asks|states|requires|instructs)|instruction|use all|all available|contradict"
T14 = [json.loads(l) for l in open(REPO / "quick_runs/results/P14/guide_trace.jsonl")]
T13 = [json.loads(l) for l in open(REPO / "quick_runs/results/P13/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T14))

Counter({'G3o': 30, 'G3n': 30, 'G3o-p': 20, 'G3n-p': 20})


In [4]:
# Stage 1: fix lessons and wrong lessons, per guide arm (runs out of 5 or 20). P13 G3 / G3-p for reference.
def stage1(trs, arm):
    row = {"arm": arm}
    for t, ids in FIX.items():
        rs = [x for x in trs if x["arm"] == arm.split(" ")[0] and x["task"] == t]
        if not rs: continue
        row[t[:7] + " fix seen"] = sum(bool(set(x["seen"]) & ids) for x in rs)
        row[t[:7] + " fix shown"] = sum(bool(set(x["shown"]) & ids) for x in rs)
        row[t[:7] + " n"] = len(rs)
    if t := [x for x in trs if x["task"] == "sf_tf03_correlation_causation" and x["arm"] == arm.split(" ")[0]]:
        row["tf03 contra shown"] = sum(bool(set(x["shown"]) & CONTRA3) for x in t)
    return row
def wrong(trs, arm):
    rs = [x for x in trs if x["arm"] == arm]
    return {"arm": arm, "runs": len(rs), "wrong seen": sum(bool(set(x["seen"]) & WRONG_IDS) for x in rs),
            "wrong accepted": sum(bool(set(x["accepted"]) & WRONG_IDS) for x in rs), "wrong shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs),
            **{f"{t[:7]} shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs if x["task"] == t) for t in TASKS4}}
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T14, "G3o"), stage1(T14, "G3n")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(T14, "G3o-p"), wrong(T14, "G3n-p")]).set_index("arm")
print(s1.to_string()); wr

     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3o                 5                  0          5                 5                  1          5                  4
G3n                 5                  2          5                 5                  0          5                  5


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3o-p,20,20,6,6,5,0,0,1
G3n-p,20,20,10,9,5,0,0,4


In [5]:
import sys, yaml
sys.path.insert(0, str(REPO / "quick_runs" / "p13")); sys.path.insert(0, str(REPO / "quick_runs" / "p12"))
import guide as GD
NF = GD.load_store("nf")
LAB = yaml.safe_load(open(REPO / "quick_runs/p17/labels_nf.yaml"))
def ids(label):
    return {i for i, L in enumerate(NF) if L.get("nf") and L["source_task"] in LAB
            and f"{L['source_run']}/{L['k']}" in (LAB[L["source_task"]][label] or [])}
REINF, FIXNF = ids("reinforcing"), ids("fix")
nf_own = Counter(L["source_task"] for L in NF if L.get("nf"))
print("nf lessons:", sum(nf_own.values()), dict(nf_own)); print("reinforcing:", len(REINF), "fix:", len(FIXNF))
T17 = [json.loads(l) for l in open(REPO / "quick_runs/results/P17/guide_trace.jsonl")]
for t in T17: t.setdefault("seen", t["shown"]); t.setdefault("accepted", t["shown"])
print(Counter(t["arm"] for t in T17))
rows = []
for t in T17:
    if t["task"] not in TASKS4: continue
    rows.append({"arm": t["arm"], "task": t["task"][:7], "run": t["run"], "n_shown": len(t["shown"]),
                 "reinf_seen": bool(set(t["seen"]) & REINF), "reinf_shown": bool(set(t["shown"]) & REINF),
                 "n_reinf_shown": len(set(t["shown"]) & REINF), "fix_shown": bool(set(t["shown"]) & FIXNF)})
ex = pd.DataFrame(rows)
print(ex.groupby("arm")[["reinf_seen", "reinf_shown", "fix_shown", "n_shown"]].sum().to_string())
ex.groupby(["arm", "task"])[["reinf_seen", "reinf_shown"]].sum().unstack()

nf lessons: 116 {'sf_tf01_goal_imbalance': 13, 'sf_tf02_player_rep_norm': 15, 'sf_tf03_correlation_causation': 15, 'sf_tf04_balance_torque_cue': 14, 'sf_tf04_balance_torque_leakfree': 15, 'sf_tf05_credit_card_leakage': 14, 'sf_tf06_metric_select_cue': 15, 'sf_tf06_metric_select_leakfree': 15}
reinforcing: 10 fix: 0
Counter({'G3-nf': 30, 'C1-nf': 30, 'E-nf': 30, 'G3d-nf': 30})
        reinf_seen  reinf_shown  fix_shown  n_shown
arm                                                
C1-nf           10           10          0      285
E-nf            10           10          0       60
G3-nf           11           10          0       57
G3d-nf          10           10          0       57


reinf_seen                         reinf_shown                        
task      sf_tf01 sf_tf02 sf_tf03 sf_tf05     sf_tf01 sf_tf02 sf_tf03 sf_tf05
arm                                                                          
C1-nf           0       5       5       0           0       5       5       0
E-nf            0       5       5       0           0       5       5       0
G3-nf           1       5       5       0           0       5       5       0
G3d-nf          0       5       5       0           0       5       5       0

In [6]:
# Guide reasons on reinforcing lessons (accepted and rejected)
rows = []
for t in T17:
    for r in t.get("rounds", []):
        for j in r.get("judgements") or []:
            try: i = int(j.get("id"))
            except (TypeError, ValueError): continue
            if i in REINF and i in r["candidates"]:
                rows.append({"arm": t["arm"], "task": t["task"][:7], "useful": j.get("useful") is True, "reason": str(j.get("reason", ""))})
jr = pd.DataFrame(rows)
print(jr.groupby(["arm", "useful"]).size().to_string())
for _, r in jr.groupby(["arm", "task", "useful"]).head(1).iterrows(): print(f"{r.arm} {r.task} useful={r.useful}: {r.reason[:240]}")

arm     useful
G3-nf   False     19
        True      15
G3d-nf  False     15
        True      15
G3-nf sf_tf01 useful=False: This is essentially a duplicate of note [614].
G3-nf sf_tf02 useful=True: Essential because the schema shows 'height_in_cm' and 'international_caps' are loaded as strings, which would prevent mean imputation and model training if not converted to numeric.
G3-nf sf_tf03 useful=True: Provides a reliable programmatic way to select numerical columns, reducing the risk of manual omission or typing errors.
G3-nf sf_tf03 useful=False: Redundant; provides the same advice as note 640.
G3-nf sf_tf06 useful=False: This note suggests caution when excluding IDs, which conflicts with the already accepted note [id 720] that explicitly instructs the agent to exclude identifiers to prevent overfitting.
G3d-nf sf_tf02 useful=True: Extremely relevant because 'height_in_cm' and 'international_caps' are listed as strings and contain non-numeric characters (e.g., '183cm', 'na'), whi

In [7]:
# Stage 2: agent runs. Targeted checks on the 4 failing tasks (passes out of 5), P14 G3n next to P12/P13 arms
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    recs = [r for r in recs if r["task_id"] in SF6]
    for r in recs:
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["sum_tokens"] = sum(c["usage"]["prompt_tokens"] + c["usage"]["completion_tokens"] for c in calls)
    return recs
ARMS = ["P12_A0", "P12_C1", "P13_E", "P13_G3", "P16_G3d", "P17_C1nf", "P17_Enf", "P17_G3nf", "P17_G3dnf"]
NAME = dict(zip(ARMS, ["A0 none", "C1 own", "E emb", "G3", "G3d", "C1-nf", "E-nf", "G3-nf", "G3d-nf"]))
recs = {a: load(a) for a in ARMS}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})
rows = []
for t, checks in TARGET.items():
    for c in checks:
        rows.append({"task": t[:7], "check": c, **{NAME[a]: sum(r["checks"].get(c) is True for r in recs[a] if r["task_id"] == t) for a in ARMS}})
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / 40
print("pooled:", pooled.round(3).to_dict()); targeted

{'P12_A0': (30, {'final_answer': 30}), 'P12_C1': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_E': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_G3': (30, {'final_answer': 29, 'max_steps': 1}), 'P16_G3d': (30, {'final_answer': 28, 'max_steps': 2}), 'P17_C1nf': (30, {'final_answer': 30}), 'P17_Enf': (30, {'final_answer': 30}), 'P17_G3nf': (30, {'final_answer': 30}), 'P17_G3dnf': (30, {'final_answer': 30})}
pooled: {'A0 none': 0.075, 'C1 own': 0.75, 'E emb': 0.475, 'G3': 0.5, 'G3d': 0.6, 'C1-nf': 0.025, 'E-nf': 0.025, 'G3-nf': 0.025, 'G3d-nf': 0.0}


,task,check,A0 none,C1 own,E emb,G3,G3d,C1-nf,E-nf,G3-nf,G3d-nf
0,sf_tf01,checks_class_distribution,0,5,5,4,2,0,1,0,0
1,sf_tf01,uses_class_weight,0,5,5,5,5,0,0,0,0
2,sf_tf01,uses_resampling,0,0,0,0,0,0,0,0,0
3,sf_tf01,uses_stratified_split,3,5,5,5,5,1,0,1,0
4,sf_tf02,handles_unit_suffix_in_height,0,4,4,4,3,0,0,0,0
5,sf_tf03,column_x_included,0,5,0,1,5,0,0,0,0
6,sf_tf03,applies_causal_reasoning,0,5,0,1,4,0,0,0,0
7,sf_tf05,used_leakage_features,0,1,0,0,0,0,0,0,0


In [8]:
# Harm, cost and exact tests
from scipy.stats import fisher_exact
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"]} for a in ARMS for r in recs[a]])
scores = sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean").round(4)
passed = (pooled * 40).round().astype(int)
fp = lambda a, b: round(fisher_exact([[a, 40 - a], [b, 40 - b]])[1], 4)
fisher = {"C1 vs C1-nf": fp(passed["C1 own"], passed["C1-nf"]), "E vs E-nf": fp(passed["E emb"], passed["E-nf"]),
          "G3 vs G3-nf": fp(passed["G3"], passed["G3-nf"]), "A0 vs C1-nf": fp(passed["A0 none"], passed["C1-nf"])}
print(fisher); scores

{'C1 vs C1-nf': np.float64(0.0), 'E vs E-nf': np.float64(0.0), 'G3 vs G3-nf': np.float64(0.0), 'A0 vs C1-nf': np.float64(0.6153)}


arm,A0 none,C1 own,C1-nf,E emb,E-nf,G3,G3-nf,G3d,G3d-nf
task,,,,,,,,,
sf_tf01_goal_imbalance,7.603000e-01,7.765000e-01,7.860000e-01,7.713000e-01,7.243000e-01,7.715000e-01,7.788000e-01,7.658000e-01,7.836000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.650909e+06,4.664201e+06,4.650909e+06,4.664201e+06,4.650909e+06,4.664201e+06,4.655340e+06,4.664201e+06
sf_tf03_correlation_causation,2.769462e+07,4.755266e+06,2.769458e+07,2.769464e+07,2.769458e+07,2.310314e+07,2.769457e+07,4.759155e+06,2.769457e+07
sf_tf04_balance_torque_leakfree,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
sf_tf05_credit_card_leakage,9.659000e-01,8.220000e-01,9.659000e-01,9.144000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01
sf_tf06_metric_select_leakfree,9.798000e-01,9.806000e-01,9.797000e-01,9.796000e-01,9.803000e-01,9.803000e-01,9.799000e-01,9.777000e-01,9.780000e-01


In [9]:
summary = {"nf_lessons": dict(nf_own), "n_reinforcing": len(REINF), "n_fix": len(FIXNF),
  "exposure": ex.groupby("arm")[["reinf_seen", "reinf_shown", "fix_shown"]].sum().reset_index().to_dict("records"),
  "exposure_by_task": ex.groupby(["arm", "task"])[["reinf_seen", "reinf_shown"]].sum().reset_index().to_dict("records"),
  "reinf_judgements": jr.groupby(["arm", "useful"]).size().reset_index(name="n").to_dict("records") if len(jr) else [],
  "runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(), "fisher_p": fisher,
  "score_mean": scores.to_dict("index")}
json.dump(summary, open(REPO / "quick_runs/results/P17/summary.json", "w"), indent=1, default=str)
"done" 

'done'